# Pharmacy Inventory Simulation: Finding the Best (s, S) Reorder Policy

**Course:** CSE 413 - Simulation & Modeling Lab

A pharmacy sells paracetamol strips. Daily demand is random (Poisson, mean 20) and an order takes 1 to 3 days to arrive.
Under an **(s, S) policy** the shop checks its *inventory position* (stock on hand + stock on order) every evening.
If the position is **at or below s**, it orders enough to bring the position **up to S**.

This notebook simulates the shop for 365 days, repeats it 100 times for each of 49 (s, S) pairs, and finds the cheapest policy.

**How to run:** click **Runtime > Run all**. The policy search takes about 1 to 2 minutes.

## Import Necessary Libraries

In [ ]:
import math, csv
import numpy as np
import scipy.stats as stats          # chi-square and t-tests (as in the lab)
import matplotlib.pyplot as plt
import matplotlib.animation as animation   # animation (as in Lab 8)
import seaborn as sns                # heatmap (as in Lab 2)
from IPython.display import Video    # show the animation video in the notebook

## Model Parameters
Change these values to try other shops. Costs are assumptions.

In [ ]:
DEMAND_MEAN = 20.0          # strips per day (Poisson)
LEAD_TIMES = (1, 2, 3)      # delivery time in days, equally likely
ORDER_COST = 300.0          # BDT per order (fixed)
HOLD_COST = 0.5             # BDT per strip per day kept in stock
SHORT_COST = 8.0            # BDT per strip of lost sale
HORIZON = 365               # days counted in one run
WARMUP = 30                 # days discarded at the start
REPS = 100                  # replications per policy

## Step 1: Random number generator (LCG)
A Linear Congruential Generator creates numbers between 0 and 1 using
`X(n+1) = (a * X(n) + increment) mod m`, with a = 1664525, increment = 1013904223, m = 2^32.

In [ ]:
class LCG:
    def __init__(self, seed=12345, a=1664525, c=1013904223, m=2**32):
        self.x, self.a, self.c, self.m = seed, a, c, m

    def next(self):
        self.x = (self.a * self.x + self.c) % self.m
        return self.x / self.m      # uniform number in [0, 1)

## Step 2: Chi-square goodness-of-fit test for the generator (as in Lab 7)
**H0:** the LCG numbers are uniformly distributed.  We split [0, 1) into 10 equal classes and compare observed and expected counts.

In [ ]:
# Step 1: Observed data (frequencies in each of 10 classes)
rng = LCG(2024)
numbers = np.array([rng.next() for _ in range(10000)])
observed, _ = np.histogram(numbers, bins=10, range=(0, 1))

# Step 2: Expected frequencies (uniform distribution)
total = np.sum(observed)
expected = np.full_like(observed, total / len(observed), dtype=float)

# Step 3: Chi-square statistic
chi_square_stat = np.sum(((observed - expected) ** 2) / expected)

# Step 4: Degrees of freedom
df = len(observed) - 1

# Step 5: Critical value and p-value
alpha = 0.05
critical_value = stats.chi2.ppf(1 - alpha, df)
p_value = 1 - stats.chi2.cdf(chi_square_stat, df)

print(f'Observed counts: {observed}')
print(f'Chi-square statistic: {chi_square_stat:.2f}')
print(f'Degrees of freedom: {df}')
print(f'Critical value (0.05 significance level): {critical_value:.3f}')
print(f'P-value: {p_value:.4f}')

if chi_square_stat > critical_value:
    print('Reject the null hypothesis. The numbers are not uniform.')
else:
    print('Fail to reject the null hypothesis. The numbers look uniformly distributed.')

In [ ]:
# Plotting observed vs expected frequencies
plt.bar(np.arange(1, 11) - 0.2, observed, width=0.4, label='Observed', alpha=0.7)
plt.bar(np.arange(1, 11) + 0.2, expected, width=0.4, label='Expected', alpha=0.7)
plt.xlabel('Class (0.0-0.1, 0.1-0.2, ...)')
plt.ylabel('Frequency')
plt.title('LCG numbers: Observed vs Expected Frequencies')
plt.legend()
plt.show()

## Step 3: Demand and lead-time generators
Demand is Poisson with mean 20, generated by the **inverse transform method**: add up the Poisson probabilities until the
cumulative probability reaches the uniform number U.  Lead time is 1, 2 or 3 days with equal probability.

In [ ]:
def poisson(rng, mean):
    u = rng.next()
    p = math.exp(-mean)      # P(D = 0)
    cdf = p
    d = 0
    while u > cdf:
        d += 1
        p *= mean / d        # P(D = d) from P(D = d-1)
        cdf += p
    return d

def lead_time(rng):
    return LEAD_TIMES[min(int(rng.next() * len(LEAD_TIMES)), len(LEAD_TIMES) - 1)]

In [ ]:
# Check the demand generator: simulate 100,000 days
r = LCG(99)
demand_sample = np.array([poisson(r, DEMAND_MEAN) for _ in range(100000)])
print('Sample mean    :', round(demand_sample.mean(), 3), '(target 20)')
print('Sample variance:', round(demand_sample.var(ddof=1), 3), '(target 20 for Poisson)')

# Histogram of simulated demand vs the true Poisson probabilities
plt.hist(demand_sample, bins=np.arange(0, 45) - 0.5, density=True, alpha=0.6, color='blue', edgecolor='white', label='Simulated')
x = np.arange(0, 45)
plt.plot(x, stats.poisson.pmf(x, DEMAND_MEAN), 'r-o', markersize=3, label='Poisson(20) theory')
plt.xlabel('Daily demand (strips)')
plt.ylabel('Density')
plt.title('Simulated daily demand vs Poisson distribution')
plt.legend()
plt.grid(True)
plt.show()

## Step 4: The inventory simulator
Each day: (1) receive arriving orders, (2) random demand arrives and is sold (unmet demand is **lost**), (3) review the inventory
position and order up to S if it is at or below s, (4) add the day's costs.
Two separate random streams (demand and lead time) with the **same seeds for every policy** make the comparison fair (common random numbers).

In [ ]:
def simulate(s, S, seed, horizon=HORIZON, warmup=WARMUP, log=None):
    dem_rng, lt_rng = LCG(seed), LCG(seed + 7919)     # two random streams
    on_hand = S
    pipeline = []                                      # outstanding orders [arrival_day, qty]
    tot = dict(order=0.0, hold=0.0, short=0.0, orders=0, demand=0, lost=0, inv=0.0)

    for day in range(1, horizon + warmup + 1):
        # 1) receive orders that arrive today
        for o in [o for o in pipeline if o[0] == day]:
            on_hand += o[1]
        pipeline = [o for o in pipeline if o[0] > day]
        start_inv = on_hand

        # 2) random demand; lost sales if stock is short
        d = poisson(dem_rng, DEMAND_MEAN)
        sold = min(on_hand, d)
        lost = d - sold
        on_hand -= sold

        # 3) review: inventory position = on hand + on order
        position = on_hand + sum(o[1] for o in pipeline)
        qty = 0
        if position <= s:
            qty = S - position
            pipeline.append([day + lead_time(lt_rng), qty])

        # 4) costs (only after the warm-up period)
        if day > warmup:
            tot["order"] += ORDER_COST if qty else 0
            tot["hold"] += HOLD_COST * on_hand
            tot["short"] += SHORT_COST * lost
            tot["orders"] += 1 if qty else 0
            tot["demand"] += d
            tot["lost"] += lost
            tot["inv"] += on_hand
            if log is not None:
                log.append((day - warmup, start_inv, d, sold, lost, on_hand, qty))

    total = tot["order"] + tot["hold"] + tot["short"]
    return {"cost_day": total / horizon, "order_day": tot["order"] / horizon,
            "hold_day": tot["hold"] / horizon, "short_day": tot["short"] / horizon,
            "service": 1 - tot["lost"] / tot["demand"], "avg_inv": tot["inv"] / horizon,
            "orders": tot["orders"]}

In [ ]:
# Try one policy for one year
one_run = simulate(30, 175, seed=1000)
for k, v in one_run.items():
    print(f'{k:10s}: {v:.3f}')

## Step 5: Replications and policy search
One run can be lucky or unlucky, so every policy is simulated **100 times** (seeds 1000 to 1099) and the results are averaged.
We test s = 20, 30, ..., 80 and S = 100, 150, 175, 200, 225, 250, 300 (49 pairs).

In [ ]:
def run_policy(s, S, reps=REPS):
    return [simulate(s, S, seed=1000 + r) for r in range(reps)]       # same seeds for all policies

def summarize(runs):
    out = {}
    for k in runs[0]:
        vals = np.array([r[k] for r in runs])
        out[k] = vals.mean()
        out[k + "_ci"] = 1.96 * vals.std(ddof=1) / np.sqrt(len(vals))   # 95% half-width
    return out

s_values = [20, 30, 40, 50, 60, 70, 80]
S_values = [100, 150, 175, 200, 225, 250, 300]

raw = {}        # all replications of each policy (used later for t-tests)
results = {}    # averages and confidence intervals
for s in s_values:
    for S in S_values:
        if S > s:
            raw[(s, S)] = run_policy(s, S)
            results[(s, S)] = summarize(raw[(s, S)])

best = min(results, key=lambda k: results[k]["cost_day"])
worst = max(results, key=lambda k: results[k]["cost_day"])
print('Policies tested:', len(results))

In [ ]:
print(f"{'(s,S)':>10} | {'Cost/day':>8} {'+-95%CI':>7} | {'Order':>6} {'Hold':>6} {'Short':>6} | {'Service%':>8} | {'AvgInv':>6}")
for k in sorted(results, key=lambda k: results[k]["cost_day"])[:8]:
    v = results[k]
    print(f"{str(k):>10} | {v['cost_day']:8.1f} {v['cost_day_ci']:7.1f} | {v['order_day']:6.1f} {v['hold_day']:6.1f} "
          f"{v['short_day']:6.1f} | {v['service']*100:8.2f} | {v['avg_inv']:6.1f}")

print()
print('Best policy (s,S) =', best, ':', round(results[best]['cost_day'], 1), 'BDT/day')
print('Worst policy (s,S) =', worst, ':', round(results[worst]['cost_day'], 1), 'BDT/day')
saving = (1 - results[best]['cost_day'] / results[worst]['cost_day']) * 100
print(f"Saving of best vs worst = {saving:.1f}%  (about {(results[worst]['cost_day'] - results[best]['cost_day']) * 365:.0f} BDT per year)")

## Step 6: Heatmap of the average cost (as in Lab 2) and service level

In [ ]:
matrix = np.array([[results[(s, S)]["cost_day"] if (s, S) in results else np.nan for S in S_values] for s in s_values])

plt.figure(figsize=(9, 5))
ax = sns.heatmap(matrix, annot=True, fmt='.0f', cmap='viridis_r', linewidths=0.5, linecolor='black',
                 xticklabels=S_values, yticklabels=s_values, cbar_kws={'label': 'BDT per day'})
ax.invert_yaxis()
ax.plot(S_values.index(best[1]) + 0.5, s_values.index(best[0]) + 0.5, 'r*', markersize=18)   # best policy
plt.xlabel('S (order-up-to level)')
plt.ylabel('s (reorder point)')
plt.title('Average cost per day (red star = best policy)')
plt.show()

In [ ]:
# Service level versus s for the best S
Sb = best[1]
ss = [s for s in s_values if (s, Sb) in results]
plt.errorbar(ss, [results[(s, Sb)]['service'] * 100 for s in ss],
             yerr=[results[(s, Sb)]['service_ci'] * 100 for s in ss], fmt='o-', capsize=3)
plt.title(f'Service level vs reorder point s (S = {Sb})')
plt.xlabel('Reorder point s')
plt.ylabel('% of demand met')
plt.grid(True)
plt.show()

## Step 7: Hypothesis tests (t-tests, as in Labs 5 and 6)
**Two-sample t-test.** Is the daily cost of the best policy different from a cautious high-stock policy (60, 200)?
Each sample has 100 replication costs.  H0: the two mean costs are equal.  We use Welch's t-test (unequal variances).

In [ ]:
data1 = np.array([r['cost_day'] for r in raw[best]])        # best policy
data2 = np.array([r['cost_day'] for r in raw[(60, 200)]])   # cautious policy

print(f'Policy {best} mean cost: {data1.mean():.2f}   Policy (60, 200) mean cost: {data2.mean():.2f}')

t_stat, p_val = stats.ttest_ind(data1, data2, equal_var=False)      # Welch's t-test

# Welch-Satterthwaite degrees of freedom and 95% confidence interval of the mean difference
v1, v2 = data1.var(ddof=1) / len(data1), data2.var(ddof=1) / len(data2)
dof = (v1 + v2) ** 2 / (v1 ** 2 / (len(data1) - 1) + v2 ** 2 / (len(data2) - 1))
diff = data1.mean() - data2.mean()
se = np.sqrt(v1 + v2)
t_crit = stats.t.ppf(0.975, dof)
ci = (diff - t_crit * se, diff + t_crit * se)

print(f't-statistic: {t_stat:.2f}')
print(f'Degrees of freedom: {dof:.2f}')
print(f'p-value: {p_val:.2e}')
print(f'Mean difference: {diff:.2f}  95% CI: [{ci[0]:.2f}, {ci[1]:.2f}]')

alpha = 0.05
if p_val < alpha:
    print('Reject the null hypothesis. The two policies have significantly different mean costs.')
else:
    print('Fail to reject the null hypothesis. The mean costs are not significantly different.')

In [ ]:
# Both policies use the same random seeds, so a paired t-test is even more precise
t_p, p_p = stats.ttest_rel(data1, data2)
print(f'Paired t-test: t = {t_p:.2f}, p = {p_p:.2e}')

plt.figure(figsize=(6, 4))
plt.bar([f'{best}', '(60, 200)'], [data1.mean(), data2.mean()], color=['blue', 'green'], alpha=0.7)
plt.errorbar([f'{best}', '(60, 200)'], [data1.mean(), data2.mean()],
             yerr=[data1.std(ddof=1), data2.std(ddof=1)], fmt='k.', capsize=5)
plt.title('Comparison of mean daily cost (error bar = std of 100 runs)')
plt.ylabel('BDT per day')
plt.grid(True)
plt.show()

**One-sample t-test.** Is the mean daily cost of the best policy *below* a target (benchmark) of 85 BDT per day?  H0: mean cost = 85.

In [ ]:
benchmark = 85
t_stat, p_val = stats.ttest_1samp(data1, benchmark, alternative='less')
print(f'Sample mean: {data1.mean():.2f}   Benchmark: {benchmark}')
print(f't-statistic: {t_stat:.2f}   Degrees of freedom: {len(data1) - 1}   p-value: {p_val:.2e}')
if p_val < 0.05:
    print(f'Reject H0. The mean daily cost is significantly below {benchmark} BDT.')
else:
    print(f'Fail to reject H0. The mean daily cost is not significantly below {benchmark} BDT.')

## Step 8: Daily dataset of the best policy (CSV)
One 365-day run of the best policy (seed 42). Each row is one day.

In [ ]:
log = []
simulate(best[0], best[1], seed=42, log=log)

with open('inventory_daily_data.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['day', 'start_inventory', 'demand', 'sales', 'lost_sales', 'end_inventory', 'order_qty'])
    w.writerows(log)

days = np.array([r[0] for r in log])
demand = np.array([r[2] for r in log])
lost = np.array([r[4] for r in log])
end_inv = np.array([r[5] for r in log])
orders = np.array([r[6] for r in log])

print('Days:', len(log))
print(f'Mean daily demand: {demand.mean():.2f} (std {demand.std(ddof=1):.2f}), min {demand.min()}, max {demand.max()}')
print(f'Total demand: {demand.sum()}   Lost sales: {lost.sum()} ({lost.sum() / demand.sum() * 100:.1f}%)')
print(f'Service level: {(1 - lost.sum() / demand.sum()) * 100:.2f}%   Stock-out days: {(lost > 0).sum()}')
print(f'Orders placed: {(orders > 0).sum()}  (mean size {orders[orders > 0].mean():.1f})')
print(f'End-of-day stock: mean {end_inv.mean():.1f}, max {end_inv.max()}')

# To download the CSV file when running in Google Colab (ignored elsewhere)
try:
    from google.colab import files
    files.download('inventory_daily_data.csv')
except Exception:
    pass

## Step 9: Animation of the stock level (as in Lab 8)
The line shows end-of-day stock for the first 60 days of the best policy; the red dot is the current day.
The video is saved with **ffmpeg** (already installed in Colab) and displayed in the notebook.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
N = 60

def update_plot(i):
    ax.clear()
    ax.plot(days[:i + 1], end_inv[:i + 1], color='b', label='End-of-day stock')
    ax.scatter(days[i], end_inv[i], color='r', s=80)                       # current day
    ax.axhline(best[0], color='orange', linestyle='--', label=f's = {best[0]}')
    ax.axhline(best[1], color='red', linestyle='--', label=f'S = {best[1]}')
    ax.set_xlim(1, N)
    ax.set_ylim(0, best[1] + 10)
    ax.set_title('Pharmacy stock level, best (s, S) policy')
    ax.set_xlabel('Day')
    ax.set_ylabel('Strips in stock')
    ax.legend(loc='upper right')

ani = animation.FuncAnimation(fig, update_plot, frames=N, interval=100)

try:
    ani.save('inventory_animation.mp4', writer='ffmpeg', fps=10)
    plt.close(fig)
    display_video = Video('inventory_animation.mp4', embed=True)
except Exception as e:
    plt.close(fig)
    print('Could not save the video (ffmpeg missing?):', e)
    display_video = None

display_video

## Conclusion
- The best policy found is **(s, S) = (30, 175)** with an average cost of about **80.7 BDT per day**; (30, 200) is just as cheap.
- The worst policy, (80, 100), costs about 231 BDT per day, so the best policy is about 65% cheaper.
- A higher reorder point s raises the service level but also the holding cost; for example (40, 200) meets about 98% of demand for only about 0.8 BDT per day more.
- The t-tests confirm that the cautious policy (60, 200) is significantly more expensive, and that the best policy's mean cost is below a target of 85 BDT per day.
- The simulated average order size (about 156 strips) agrees with the economic order quantity sqrt(2 K lambda / h) = 155 strips.

**Limitations:** costs are assumed, demand is Poisson with constant mean, unmet demand is lost, and only one product is modeled.